# Whirlpool AI Operations Hub: Multi-Agent & BigQuery Vector Search
### Demonstração Prática de IA Generativa, Governança e Busca Vetorial no Google Cloud

Este notebook demonstra o fluxo completo da solução desenvolvida sob medida para a vaga de **AI Analyst** da **Whirlpool**:
1. **Conexão com Google Cloud (Vertex AI e BigQuery)**
2. **Governança de Dados (Compliance PULSE / PIA / LGPD)**
3. **Segmentação e Ingestão Multimodal com Gemini no Vertex AI**
4. **Geração de Embeddings (`text-embedding-004`) e BigQuery Vector Search**
5. **Consultas RAG de Alto Impacto para Tomada de Decisão**
6. **Geração Automatizada de Diagramas de Processos (Mermaid.js)**

## 1. Inicialização e Teste de Conexões

In [ ]:
import sys
sys.path.append("..")

from src.utils.gcp_client import test_connections, PROJECT_ID, LOCATION
print(f"Projeto Ativo: {PROJECT_ID} ({LOCATION})")
test_connections()

## 2. Governança de Dados: Mascaramento de PII (Compliance PULSE / PIA)
Antes de qualquer persistência no BigQuery, o `GovernanceAgent` audita o texto e mascara CPFs, matrículas funcionais e dados sensíveis.

In [ ]:
from data.sample_meetings import SAMPLE_MEETINGS
from src.agents.governance_agent import GovernanceAgent

sample = SAMPLE_MEETINGS[0]
print("--- TEXTO ORIGINAL (COM PII) ---")
print(sample["raw_text"][:350] + "...")

gov = GovernanceAgent()
result = gov.sanitize_transcript(sample["raw_text"])

print("\n--- AUDITORIA DE GOVERNANÇA ---")
print(f"Métricas: {result['metrics']}")
print(f"Status: {result['compliance_status']}")

print("\n--- TEXTO SANITIZADO ---")
print(result["sanitized_text"][:350] + "...")

## 3. Ingestão e Estruturação Multimodal (Vertex AI)
O `MultimodalMeetingAgent` analisa a discussão e divide o diálogo em chunks semânticos com participantes, tópicos e ações.

In [ ]:
from src.agents.multimodal_agent import MultimodalMeetingAgent

agent = MultimodalMeetingAgent()
structured = agent.process_transcript_text(
    meeting_id=sample["meeting_id"],
    title=sample["meeting_title"],
    raw_text=result["sanitized_text"],
    department=sample["department"],
    date=sample["meeting_date"]
)

print(f"Reunião: {structured['meeting_title']}")
print(f"Resumo Executivo: {structured.get('summary')}")
print(f"Decisões Principais: {structured.get('key_decisions')}")
print(f"Total de Chunks gerados: {len(structured.get('chunks', []))}")

## 4. Geração de Embeddings e BigQuery Vector Search
Executamos uma busca semântica direta no BigQuery usando a função `VECTOR_SEARCH` e similaridade de cosseno.

In [ ]:
from src.pipeline.embeddings import generate_query_embedding
from src.pipeline.bigquery_loader import search_similar_chunks

pergunta = "Qual alternativa foi encontrada para os compressores de Rio Claro e quanto custará o frete?"
query_vector = generate_query_embedding(pergunta)

hits = search_similar_chunks(query_vector, top_k=3)
print(f"Resultados recuperados do BigQuery para a pergunta: '{pergunta}'\n")
for i, hit in enumerate(hits, 1):
    print(f"[{i}] Similaridade: {hit['similarity_score']} | Falante: {hit['speaker']} | Dept: {hit['department']}")
    print(f"    Conteúdo: {hit['content']}")
    print("-" * 70)

## 5. Agente Consultor RAG (Síntese Executiva)
O `RagConsultantAgent` sintetiza a resposta final com embasamento total nos trechos recuperados do BigQuery.

In [ ]:
from src.agents.rag_agent import RagConsultantAgent

rag = RagConsultantAgent()
resposta = rag.answer_query("O que foi decidido sobre os compressores de Rio Claro e qual o impacto financeiro?")

print("RESPOSTA DO AGENTE RAG:")
print(resposta["answer"])
print("\nFONTES AUDITADAS:")
for s in resposta["sources"]:
    print(f"- {s['meeting_title']} (Falante: {s['speaker']} | Score: {s['similarity_score']})")

## 6. Geração de Entregáveis de Processo: Diagramas Mermaid & Matriz RACI
O `ProcessDiagramAgent` mapeia o fluxo operacional da decisão e estrutura a matriz de governança de papéis.

In [ ]:
from src.agents.diagram_agent import ProcessDiagramAgent

diagram_agent = ProcessDiagramAgent()
mermaid_code = diagram_agent.generate_operational_diagram(structured)
raci_table = diagram_agent.generate_raci_matrix(structured)

print("--- DIAGRAMA MERMAID GERADO ---")
print(mermaid_code)
print("\n--- MATRIZ RACI ---")
print(raci_table)